In [ ]:
import os
from pathlib import Path

import pandas as pd
from tokenizers import (
    ByteLevelBPETokenizer,
)
from tokenizers.decoders import ByteLevel
from transformers import (
    DataCollatorForLanguageModeling,
    LineByLineTextDataset,
    pipeline,
    RobertaConfig,
    RobertaForMaskedLM,
    RobertaTokenizer,
    Trainer, 
    TrainingArguments,
)
import torch


In [2]:
DEVICE = (torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))
print(f'Training on device {DEVICE}')

Training on device mps


In [3]:
df = pd.read_csv('./echo_from_epikriz_corpus.csv')

df.head(2)

,echo_from_epikriz,target
0,плахова клименко увеличен тонкие подвижные изм...,0.0
1,левое предсердие особенности увеличено левый ж...,0.0


In [26]:
df['echo_from_epikriz'].to_csv(
    'data.txt',
    index=False,
    header=False,
    )

In [28]:
paths = [str(x) for x in Path('.').glob('**/*.txt')]

paths

['EchoBERT/data.txt']

In [ ]:
tokenizer = ByteLevelBPETokenizer()

# Customize training
tokenizer.train(
    files=paths, 
    vocab_size=32000,
    min_frequency=2,
    show_progress=True,
    special_tokens=[
        '<s>',
        '<pad>',
        '</s>',
        '<unk>',
        '<mask>',
       ],
)

In [30]:
token_dir = './EchoBERT'
if not os.path.exists(token_dir):
  os.makedirs(token_dir)
tokenizer.save_model('EchoBERT')

['EchoBERT/vocab.json', 'EchoBERT/merges.txt']

In [31]:
tokenizer = ByteLevelBPETokenizer(
    './EchoBERT/vocab.json',
    './EchoBERT/merges.txt',
    )

In [ ]:
decoder = ByteLevel()
decoder.decode([ 'ĠÐ¿ÑĢÐµÐ´ÑģÐµÑĢÐ´Ð¸Ðµ' ])

' предсердие'

In [35]:
decoder.decode(tokenizer.encode('левое предсердие увеличено левый').tokens)

'левое предсердие увеличено левый'

In [37]:
tokenizer.encode('холманская левые отделы сердца увеличены')

Encoding(num_tokens=5, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [38]:
tokenizer.enable_truncation(max_length=512)

In [39]:
config = RobertaConfig(
    vocab_size=52_000,
    max_position_embeddings=514,
    num_attention_heads=12,
    num_hidden_layers=6,
    type_vocab_size=1,
)

print(config)

RobertaConfig {
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.43.2",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 52000
}



In [40]:
tokenizer = RobertaTokenizer.from_pretrained('./EchoBERT', max_length=512)

In [41]:
model = RobertaForMaskedLM(config=config).to(DEVICE)

print(model)
print(model.num_parameters())

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): La

In [266]:
LP = list(model.parameters())
lp = len(LP)

print(lp)

for p in range(0, lp):
  print(LP[p])

106
Parameter containing:
tensor([[-4.0852e-02,  1.2715e-04,  1.3338e-02,  ..., -2.8340e-03,
          8.0503e-03,  1.3501e-02],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [-2.0155e-02, -2.0460e-02, -3.9781e-03,  ...,  3.2066e-04,
         -6.1199e-03,  2.4487e-03],
        ...,
        [-1.3111e-02,  6.5614e-03,  6.8076e-03,  ..., -1.4099e-02,
         -1.2134e-02,  8.4553e-05],
        [-2.4775e-02,  9.5362e-03,  3.5227e-02,  ..., -1.9373e-03,
          1.1777e-02,  1.6380e-02],
        [-2.2680e-02,  3.1933e-02, -2.4312e-02,  ...,  2.8271e-02,
         -1.6353e-03, -2.4188e-02]], device='mps:0', requires_grad=True)
Parameter containing:
tensor([[ 0.0109, -0.0211,  0.0244,  ..., -0.0091,  0.0181,  0.0051],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0054, -0.0004, -0.0107,  ...,  0.0043, -0.0292, -0.0358],
        ...,
        [-0.0238,  0.0195,  0.0203,  ..., -0.0223, -0.0171,  

In [267]:
np = 0

for p in range(0, lp):#number of tensors
  PL2 = True
  try:
    L2 = len(LP[p][0]) #check if 2D
  except:
    L2 = 1             #not 2D but 1D
    PL2 = False
  L1 = len(LP[p])      
  L3 = L1 * L2
  
  np += L3             # number of parameters per tensor
  if PL2==True:
    print(p, L1, L2, L3)  # displaying the sizes of the parameters
  if PL2==False:
    print(p, L1, L3)  # displaying the sizes of the parameters

print(np)     

0 32000 768 24576000
1 514 768 394752
2 1 768 768
3 768 768
4 768 768
5 768 768 589824
6 768 768
7 768 768 589824
8 768 768
9 768 768 589824
10 768 768
11 768 768 589824
12 768 768
13 768 768
14 768 768
15 3072 768 2359296
16 3072 3072
17 768 3072 2359296
18 768 768
19 768 768
20 768 768
21 768 768 589824
22 768 768
23 768 768 589824
24 768 768
25 768 768 589824
26 768 768
27 768 768 589824
28 768 768
29 768 768
30 768 768
31 3072 768 2359296
32 3072 3072
33 768 3072 2359296
34 768 768
35 768 768
36 768 768
37 768 768 589824
38 768 768
39 768 768 589824
40 768 768
41 768 768 589824
42 768 768
43 768 768 589824
44 768 768
45 768 768
46 768 768
47 3072 768 2359296
48 3072 3072
49 768 3072 2359296
50 768 768
51 768 768
52 768 768
53 768 768 589824
54 768 768
55 768 768 589824
56 768 768
57 768 768 589824
58 768 768
59 768 768 589824
60 768 768
61 768 768
62 768 768
63 3072 768 2359296
64 3072 3072
65 768 3072 2359296
66 768 768
67 768 768
68 768 768
69 768 768 589824
70 768 768
71 768 768

In [43]:
dataset = LineByLineTextDataset(
    tokenizer=tokenizer,
    file_path='./data.txt',
    block_size=128,
)

In [44]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

In [45]:
training_args = TrainingArguments(
    output_dir='./EchoBERT',
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=64,
    save_steps=10_000,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

In [46]:
%%time
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/106 [00:00<?, ?it/s]

{'train_runtime': 947.5366, 'train_samples_per_second': 7.102, 'train_steps_per_second': 0.112, 'train_loss': 8.30653870780513, 'epoch': 1.0}
CPU times: user 9.6 s, sys: 4min 15s, total: 4min 25s
Wall time: 15min 47s


TrainOutput(global_step=106, training_loss=8.30653870780513, metrics={'train_runtime': 947.5366, 'train_samples_per_second': 7.102, 'train_steps_per_second': 0.112, 'total_flos': 223098012379008.0, 'train_loss': 8.30653870780513, 'epoch': 1.0})

In [47]:
trainer.save_model('./EchoBERT')

In [48]:
fill_mask = pipeline(
    'fill-mask',
    model='./EchoBERT',
    tokenizer='./EchoBERT'
)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [55]:
fill_mask('изменен норме изменен норме дуга аорты левая правые отделы умеренно увеличены<mask>.')

[{'score': 0.020342586562037468,
  'token': 347,
  'token_str': ' изменен',
  'sequence': 'изменен норме изменен норме дуга аорты левая правые отделы умеренно увеличены изменен.'},
 {'score': 0.017404409125447273,
  'token': 344,
  'token_str': ' створки',
  'sequence': 'изменен норме изменен норме дуга аорты левая правые отделы умеренно увеличены створки.'},
 {'score': 0.010793433524668217,
  'token': 345,
  'token_str': ' клапан',
  'sequence': 'изменен норме изменен норме дуга аорты левая правые отделы умеренно увеличены клапан.'},
 {'score': 0.010713540017604828,
  'token': 395,
  'token_str': ' норма',
  'sequence': 'изменен норме изменен норме дуга аорты левая правые отделы умеренно увеличены норма.'},
 {'score': 0.008686392568051815,
  'token': 444,
  'token_str': ' заключение',
  'sequence': 'изменен норме изменен норме дуга аорты левая правые отделы умеренно увеличены заключение.'}]